In [5]:
import itertools

# A helper class to manage node creation.
class NodeFactory:
    def __init__(self, start=0):
        self.counter = start
    
    def new_node(self):
        n = self.counter
        self.counter += 1
        return n

def share_exactly_one_node(edgeA, edgeB, edge_endpoints):
    """
    Return True if edgeA and edgeB share exactly one endpoint,
    given a partial (or complete) assignment in edge_endpoints.
    Assumes edgeA and edgeB are already assigned.
    """
    (a1, a2) = edge_endpoints[edgeA]
    (b1, b2) = edge_endpoints[edgeB]
    
    shared = len({a1, a2}.intersection({b1, b2}))
    return (shared == 1)

def assign_edge_consistent_with(edge_to_assign, assigned_edge, edge_endpoints, node_factory):
    """
    Assign edge_to_assign so that it shares exactly one endpoint with assigned_edge.
    - If edge_to_assign is already assigned, check consistency and return True/False.
    - If not assigned, pick one endpoint from assigned_edge to share, 
      and create a new node for the other endpoint.
    This function tries all ways to share one endpoint and stops as soon as one works.
    
    Returns True if assignment is done (or consistent), False if impossible.
    """
    if edge_to_assign in edge_endpoints:
        # Already assigned: just check it shares exactly one endpoint with assigned_edge
        return share_exactly_one_node(edge_to_assign, assigned_edge, edge_endpoints)
    
    # Not assigned yet: we must pick exactly one endpoint to share.
    (a1, a2) = edge_endpoints[assigned_edge]
    # Try each endpoint of assigned_edge as the shared node.
    for shared_node in (a1, a2):
        other_node = node_factory.new_node()
        
        # Attempt assignment
        edge_endpoints[edge_to_assign] = (min(shared_node, other_node), 
                                          max(shared_node, other_node))
        
        # Check consistency
        if share_exactly_one_node(edge_to_assign, assigned_edge, edge_endpoints):
            return True
        else:
            # revert assignment, try next
            del edge_endpoints[edge_to_assign]
            # we do NOT revert the node_factory counter, so this can create "extra" node IDs
            # in practice, that doesn't affect correctness, just node naming.
    
    return False

def try_realize_cycle(cycle, edge_endpoints, node_factory):
    """
    Attempt to embed all edges in 'cycle' as a ring in the partial graph 'edge_endpoints'.
    We try every permutation of edges in this cycle to see if we can form a consistent loop.
    If successful, we update edge_endpoints in-place and return True.
    If not, we revert any partial assignments and return False.
    """
    # We'll need to back up the current state in case we fail.
    backup_state = dict(edge_endpoints)
    
    cycle_edges = list(cycle)
    
    # Try every permutation to see if we can form a ring
    for perm in itertools.permutations(cycle_edges):
        # Restore the backup state before each attempt
        edge_endpoints.clear()
        edge_endpoints.update(backup_state)
        
        # 1) Assign the first edge if unassigned, simply create two new nodes for it.
        first_edge = perm[0]
        if first_edge not in edge_endpoints:
            n1 = node_factory.new_node()
            n2 = node_factory.new_node()
            edge_endpoints[first_edge] = (min(n1, n2), max(n1, n2))
        
        # 2) Try to chain the rest of edges in perm
        success = True
        for i in range(len(perm) - 1):
            e_curr = perm[i]
            e_next = perm[i+1]
            # Try to ensure e_next shares exactly one endpoint with e_curr
            if not assign_edge_consistent_with(e_next, e_curr, edge_endpoints, node_factory):
                success = False
                break
        
        if success:
            # Finally, check that the last edge shares exactly one node with the first edge
            last_edge = perm[-1]
            if share_exactly_one_node(first_edge, last_edge, edge_endpoints):
                # We have a consistent ring for this permutation!
                return True
            else:
                success = False
        
        # If we failed at some point, continue trying other permutations
    # If no permutation worked, revert to backup and return False
    edge_endpoints.clear()
    edge_endpoints.update(backup_state)
    return False

def build_topology(cycles):
    """
    Main driver function.
    cycles is a list of sets (or lists) of edge IDs.
    
    Returns a dict {edge_id: (nodeA, nodeB)} if successful, or None if no solution.
    """
    edge_endpoints = {}
    node_factory = NodeFactory(start=0)
    
    for cycle in cycles:
        # Try to embed this cycle in the current partial graph.
        ok = try_realize_cycle(cycle, edge_endpoints, node_factory)
        if not ok:
            return None  # no solution
    
    return edge_endpoints

if __name__ == "__main__":
    # Example usage:
    cycles = [
        {1, 2, 3, 4, 5, 6, 7, 8}  ]
    
    solution = build_topology(cycles)
    if solution is None:
        print("No feasible solution found.")
    else:
        print("Found a feasible solution!")
        for e, (n1, n2) in sorted(solution.items()):
            print(f"Edge {e} -> NodePair({n1}, {n2})")


Found a feasible solution!
Edge 1 -> NodePair(0, 1)
Edge 2 -> NodePair(0, 2)
Edge 3 -> NodePair(0, 3)
Edge 4 -> NodePair(0, 4)
Edge 5 -> NodePair(0, 5)
Edge 6 -> NodePair(0, 6)
Edge 7 -> NodePair(0, 7)
Edge 8 -> NodePair(0, 8)
